# Stage 6 — Multi-Level CTC Model Architecture

This notebook runs a tiny Wav2Vec2-BERT model with synthetic feature frames. It does not train or download pretrained weights. Raw Stage 5 waveforms are transformed to Wav2Vec2-BERT `input_features` by the matching Hugging Face feature extractor, as in the original inference path.

```text
16 kHz mono waveform → HF feature extractor (160 values/frame) → Wav2Vec2-BERT encoder + adapter
      ├─ phonemes (43 classes)
      └─ ten independent sifat CTC heads (3 or 4 classes each)
```

Phoneme labels are symbols; each sifat target represents a phoneme group. ID 0 is the CTC blank/pad class. `-100` is ignored target padding and is not the blank.

In [1]:
import sys
from pathlib import Path
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'quran_model').is_dir())
sys.path.insert(0, str(ROOT / 'src'))

import torch
from transformers import Wav2Vec2BertConfig
from quran_model import QuranMultiLevelCTC, QuranMultiLevelCTCConfig


## 1. Initialize a small model

The encoder width and depth are reduced for a fast demonstration. The encoder remains Wav2Vec2-BERT, with the checkpoint adapter enabled.

In [2]:
tiny_encoder = Wav2Vec2BertConfig(
    hidden_size=16, num_hidden_layers=1, num_attention_heads=4,
    intermediate_size=32, feature_projection_input_dim=160,
    conv_depthwise_kernel_size=3, conformer_conv_dropout=0.0,
    hidden_dropout=0.0, attention_dropout=0.0, activation_dropout=0.0,
    feat_proj_dropout=0.0, layerdrop=0.0, apply_spec_augment=False,
    left_max_position_embeddings=8, right_max_position_embeddings=4,
    add_adapter=True, adapter_kernel_size=3, adapter_stride=2,
    num_adapter_layers=1,
)
model = QuranMultiLevelCTC(QuranMultiLevelCTCConfig(encoder_config=tiny_encoder)).eval()
print(type(model.encoder).__name__, 'adapter:', model.encoder.config.add_adapter)


Wav2Vec2BertModel adapter: True


## 2. Synthetic features and all output shapes

Direct model input is `(batch, feature_frames, 160)`, matching `AutoFeatureExtractor` output. The adapter downsamples the 20-frame padded tensor to 10 output frames. The mask marks valid input feature frames; `output_lengths` reports each item's valid CTC frames.

In [3]:
features = torch.randn(2, 20, 160)
mask = torch.tensor([[1] * 20, [1] * 16 + [0] * 4])
with torch.no_grad():
    output = model(input_features=features, attention_mask=mask)
print('valid output lengths:', output.output_lengths.tolist())
for level, logits in output.logits.items():
    print(f'{level:24s} {tuple(logits.shape)}')


valid output lengths: [10, 8]
phonemes                 (2, 10, 43)
hams_or_jahr             (2, 10, 3)
shidda_or_rakhawa        (2, 10, 4)
tafkheem_or_taqeeq       (2, 10, 4)
itbaq                    (2, 10, 3)
safeer                   (2, 10, 3)
qalqla                   (2, 10, 3)
tikraar                  (2, 10, 3)
tafashie                 (2, 10, 3)
istitala                 (2, 10, 3)
ghonna                   (2, 10, 3)


## 3. Relationship between levels

All heads share encoder frames but produce independent CTC streams. The phoneme head emits a character-level phonetic sequence. Each sifat head emits one categorical property per phoneme group. Original inference decodes these streams and aligns sifat groups to phoneme groups; independent heads do not guarantee one-to-one decoded alignment.

The output above demonstrates all eleven configured heads. No weights were trained; parameters are randomly initialized.